In [ ]:
# Fabric Notebook — 02_silver_cleanse
# Bronze (raw CSVs) -> Silver (cleansed, conformed, typed, deduped)
#
# Reads raw Delta tables from the bronze lakehouse, applies data-quality
# rules, standardizes types and units, removes duplicates, and writes
# clean Delta tables to the silver lakehouse.

from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import (
    IntegerType, DoubleType, BooleanType, StringType, TimestampType, DateType
)

# ---- Config: lakehouse / schema ----
# Source and target live in the same lakehouse, different schemas.
LAKEHOUSE = "agri_lakehouse"
BRONZE_SCHEMA = "bronzedb"   # source schema
SILVER_SCHEMA = "silverdb"   # target schema
TABLE_SUFFIX = "_dim"        # both bronze and silver tables use the <name>_dim convention

def bronze(name):
    return spark.read.table(f"{LAKEHOUSE}.{BRONZE_SCHEMA}.{name}{TABLE_SUFFIX}")

def write_silver(df, name):
    target = f"{LAKEHOUSE}.{SILVER_SCHEMA}.{name}{TABLE_SUFFIX}"
    (df.write.format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(target))
    print(f"  wrote {target}: {df.count():,} rows")

# A reusable helper to dedupe by a key, keeping the most recent record
def dedupe_keep_latest(df, keys, order_col):
    w = Window.partitionBy(*keys).orderBy(F.col(order_col).desc())
    return (df.withColumn("_rn", F.row_number().over(w))
              .filter(F.col("_rn") == 1)
              .drop("_rn"))

print("=== SILVER LAYER ===")

# -------------------------------------------------------------------
# 1. farmers
# -------------------------------------------------------------------
print("Cleansing farmers...")
farmers = (
    bronze("farmers")
    .withColumn("farmer_id", F.col("farmer_id").cast(IntegerType()))
    .withColumn("name", F.trim(F.col("name")))
    .withColumn("district", F.initcap(F.trim(F.col("district"))))
    # Normalize organic flag to a clean boolean + keep the label
    .withColumn("is_organic",
                F.when(F.lower(F.trim(F.col("organic_certification"))) == "certified organic", F.lit(True))
                 .otherwise(F.lit(False)))
    .withColumn("organic_certification", F.trim(F.col("organic_certification")))
    # Drop rows missing a key
    .filter(F.col("farmer_id").isNotNull() & (F.col("name") != ""))
)
# Keep valid districts only
VALID_DISTRICTS = ["Ben Tre", "Tra Vinh", "Tien Giang", "Vinh Long"]
farmers = farmers.filter(F.col("district").isin(VALID_DISTRICTS))
farmers = dedupe_keep_latest(farmers, ["farmer_id"], "farmer_id")
write_silver(farmers, "farmers")

# -------------------------------------------------------------------
# 2. lots
# -------------------------------------------------------------------
print("Cleansing lots...")
lots = (
    bronze("lots")
    .withColumn("lot_id", F.trim(F.col("lot_id")))
    .withColumn("farmer_id", F.col("farmer_id").cast(IntegerType()))
    .withColumn("intake_date", F.to_date(F.col("intake_date"), "yyyy-MM-dd"))
    .withColumn("collection_point", F.trim(F.col("collection_point")))
    .withColumn("district", F.initcap(F.trim(F.col("district"))))
    .withColumn("weight_kg", F.col("weight_kg").cast(DoubleType()))
    .withColumn("brix", F.col("brix").cast(DoubleType()))
    .withColumn("grade", F.upper(F.trim(F.col("grade"))))
    .withColumn("is_organic", F.col("is_organic").cast(BooleanType()))
)
# Data-quality rules: valid ranges and allowed values
lots = (
    lots
    .filter(F.col("lot_id").isNotNull())
    .filter(F.col("farmer_id").isNotNull())
    .filter(F.col("intake_date").isNotNull())
    .filter((F.col("weight_kg") > 0) & (F.col("weight_kg") <= 2000))   # drop impossible weights
    .filter((F.col("brix") >= 0) & (F.col("brix") <= 15))              # drop impossible brix
    .filter(F.col("grade").isin("A", "B", "C"))
)
# Referential integrity: lot must reference an existing farmer
valid_farmer_ids = farmers.select("farmer_id").distinct()
lots = lots.join(F.broadcast(valid_farmer_ids), "farmer_id", "inner")
# Dedupe by lot_id
lots = dedupe_keep_latest(lots, ["lot_id"], "intake_date")
# Add a quality_flag derived column for convenience
lots = lots.withColumn(
    "quality_flag",
    F.when(F.col("brix") < 5.0, F.lit("Low Brix"))
     .when(F.col("grade") == "C", F.lit("Low Grade"))
     .otherwise(F.lit("OK"))
)
write_silver(lots, "lots")

# -------------------------------------------------------------------
# 3. production_runs
# -------------------------------------------------------------------
print("Cleansing production_runs...")
runs = (
    bronze("production_runs")
    .withColumn("run_id", F.trim(F.col("run_id")))
    .withColumn("plant", F.trim(F.col("plant")))
    .withColumn("line", F.trim(F.col("line")))
    .withColumn("product", F.trim(F.col("product")))
    .withColumn("shift", F.initcap(F.trim(F.col("shift"))))
    .withColumn("start_datetime", F.to_timestamp(F.col("start_datetime")))
    .withColumn("end_datetime", F.to_timestamp(F.col("end_datetime")))
    .withColumn("duration_hours", F.col("duration_hours").cast(DoubleType()))
    .withColumn("input_weight_kg", F.col("input_weight_kg").cast(DoubleType()))
    .withColumn("expected_yield", F.col("expected_yield").cast(DoubleType()))
    .withColumn("actual_yield", F.col("actual_yield").cast(DoubleType()))
    .withColumn("yield_pct", F.col("yield_pct").cast(DoubleType()))
    .withColumn("avg_brix", F.col("avg_brix").cast(DoubleType()))
    .withColumn("grade_a_pct", F.col("grade_a_pct").cast(DoubleType()))
    .withColumn("downtime_minutes", F.col("downtime_minutes").cast(IntegerType()))
    .withColumn("is_organic_run", F.col("is_organic_run").cast(BooleanType()))
    .withColumn("num_lots_consumed", F.col("num_lots_consumed").cast(IntegerType()))
)
runs = (
    runs
    .filter(F.col("run_id").isNotNull())
    .filter(F.col("start_datetime").isNotNull())
    .filter(F.col("expected_yield") > 0)
    .filter((F.col("yield_pct") >= 0) & (F.col("yield_pct") <= 100))
)
# Recompute yield_pct defensively (don't trust the source)
runs = runs.withColumn(
    "yield_pct",
    F.round(F.col("actual_yield") / F.col("expected_yield") * 100, 2)
)
# Derived: a run date (date only) for joining to dim_date
runs = runs.withColumn("run_date", F.to_date(F.col("start_datetime")))
# Derived: yield band for quick filtering in the demo
runs = runs.withColumn(
    "yield_band",
    F.when(F.col("yield_pct") >= 90, F.lit("High (>=90%)"))
     .when(F.col("yield_pct") >= 80, F.lit("Normal (80-90%)"))
     .otherwise(F.lit("Low (<80%)"))
)
runs = dedupe_keep_latest(runs, ["run_id"], "start_datetime")
write_silver(runs, "production_runs")

# -------------------------------------------------------------------
# 4. lot_consumption (bridge)
# -------------------------------------------------------------------
print("Cleansing lot_consumption...")
consumption = (
    bronze("lot_consumption")
    .withColumn("run_id", F.trim(F.col("run_id")))
    .withColumn("lot_id", F.trim(F.col("lot_id")))
    .withColumn("farmer_id", F.col("farmer_id").cast(IntegerType()))
    .withColumn("weight_consumed", F.col("weight_consumed").cast(DoubleType()))
    .filter(F.col("run_id").isNotNull() & F.col("lot_id").isNotNull())
    .filter(F.col("weight_consumed") > 0)
)
# Referential integrity: keep only rows whose run + lot exist in silver
valid_runs = runs.select("run_id").distinct()
valid_lots = lots.select("lot_id").distinct()
consumption = (
    consumption
    .join(F.broadcast(valid_runs), "run_id", "inner")
    .join(valid_lots, "lot_id", "inner")
)
# Dedupe composite key
consumption = consumption.dropDuplicates(["run_id", "lot_id"])
write_silver(consumption, "lot_consumption")

# -------------------------------------------------------------------
# 5. quality_tests
# -------------------------------------------------------------------
print("Cleansing quality_tests...")
quality = (
    bronze("quality_tests")
    .withColumn("test_id", F.trim(F.col("test_id")))
    .withColumn("run_id", F.trim(F.col("run_id")))
    .withColumn("test_name", F.trim(F.col("test_name")))
    .withColumn("category", F.initcap(F.trim(F.col("category"))))
    .withColumn("value", F.col("value").cast(DoubleType()))
    .withColumn("target", F.col("target").cast(DoubleType()))
    .withColumn("tolerance", F.col("tolerance").cast(DoubleType()))
    .withColumn("unit", F.trim(F.col("unit")))
    .withColumn("pass_fail", F.initcap(F.trim(F.col("pass_fail"))))
    .withColumn("test_datetime", F.to_timestamp(F.col("test_datetime")))
    .filter(F.col("test_id").isNotNull() & F.col("run_id").isNotNull())
    .filter(F.col("pass_fail").isin("Pass", "Fail"))
)
# Referential integrity to runs
quality = quality.join(F.broadcast(valid_runs), "run_id", "inner")
# Defensive recompute of pass/fail from value vs target+/-tolerance
quality = quality.withColumn(
    "pass_fail_calc",
    F.when(
        (F.col("value") >= F.col("target") - F.col("tolerance")) &
        (F.col("value") <= F.col("target") + F.col("tolerance")),
        F.lit("Pass")
    ).otherwise(F.lit("Fail"))
)
quality = quality.withColumn("test_date", F.to_date(F.col("test_datetime")))
quality = dedupe_keep_latest(quality, ["test_id"], "test_datetime")
write_silver(quality, "quality_tests")

# -------------------------------------------------------------------
# 6. sales_orders
# -------------------------------------------------------------------
print("Cleansing sales_orders...")
sales = (
    bronze("sales_orders")
    .withColumn("order_id", F.trim(F.col("order_id")))
    .withColumn("order_date", F.to_date(F.col("order_date"), "yyyy-MM-dd"))
    .withColumn("customer", F.trim(F.col("customer")))
    .withColumn("country", F.trim(F.col("country")))
    .withColumn("region", F.trim(F.col("region")))
    .withColumn("channel", F.trim(F.col("channel")))
    .withColumn("is_domestic", F.col("is_domestic").cast(BooleanType()))
    .withColumn("is_organic", F.col("is_organic").cast(BooleanType()))
    .withColumn("run_id", F.trim(F.col("run_id")))
    .withColumn("product", F.trim(F.col("product")))
    .withColumn("quantity_units", F.col("quantity_units").cast(IntegerType()))
    .withColumn("unit_price_usd", F.col("unit_price_usd").cast(DoubleType()))
    .withColumn("revenue_usd", F.col("revenue_usd").cast(DoubleType()))
    .withColumn("revenue_local", F.col("revenue_local").cast(DoubleType()))
    .withColumn("currency", F.upper(F.trim(F.col("currency"))))
    .withColumn("margin_pct", F.col("margin_pct").cast(DoubleType()))
    .withColumn("margin_usd", F.col("margin_usd").cast(DoubleType()))
    .withColumn("on_time_delivery", F.col("on_time_delivery").cast(BooleanType()))
    .filter(F.col("order_id").isNotNull() & F.col("order_date").isNotNull())
    .filter(F.col("quantity_units") > 0)
    .filter(F.col("revenue_usd") >= 0)
)
# Referential integrity to runs
sales = sales.join(F.broadcast(valid_runs), "run_id", "inner")
sales = dedupe_keep_latest(sales, ["order_id"], "order_date")
write_silver(sales, "sales_orders")

# -------------------------------------------------------------------
# 7. inventory_snapshots (daily finished-goods stock position)
# -------------------------------------------------------------------
print("Cleansing inventory_snapshots...")
VALID_STATUSES = ["OK", "LOW", "CRITICAL", "OUT"]
inventory = (
    bronze("inventory_snapshots")
    .withColumn("snapshot_date", F.to_date(F.col("snapshot_date"), "yyyy-MM-dd"))
    .withColumn("product", F.trim(F.col("product")))
    .withColumn("product_family", F.trim(F.col("product_family")))
    .withColumn("warehouse", F.trim(F.col("warehouse")))
    .withColumn("region", F.trim(F.col("region")))
    .withColumn("opening_units", F.col("opening_units").cast(IntegerType()))
    .withColumn("receipts_units", F.col("receipts_units").cast(IntegerType()))
    .withColumn("issues_units", F.col("issues_units").cast(IntegerType()))
    .withColumn("on_hand_units", F.col("on_hand_units").cast(IntegerType()))
    .withColumn("avg_daily_demand", F.col("avg_daily_demand").cast(DoubleType()))
    .withColumn("lead_time_days", F.col("lead_time_days").cast(IntegerType()))
    .withColumn("safety_stock_units", F.col("safety_stock_units").cast(IntegerType()))
    .withColumn("reorder_point_units", F.col("reorder_point_units").cast(IntegerType()))
    .withColumn("on_order_units", F.col("on_order_units").cast(IntegerType()))
    .withColumn("days_of_supply", F.col("days_of_supply").cast(DoubleType()))
    .withColumn("stock_status", F.upper(F.trim(F.col("stock_status"))))
    .withColumn("restock_alert", F.col("restock_alert").cast(BooleanType()))
    .withColumn("recommended_order_units", F.col("recommended_order_units").cast(IntegerType()))
)
# Data-quality rules: keys present, non-negative balances, valid status
inventory = (
    inventory
    .filter(F.col("snapshot_date").isNotNull())
    .filter((F.col("product") != "") & (F.col("warehouse") != ""))
    .filter(F.col("on_hand_units") >= 0)
    .filter(F.col("reorder_point_units") >= 0)
    .filter(F.col("stock_status").isin(VALID_STATUSES))
)
# Referential integrity: product must exist in the sales/production universe
valid_products = (
    runs.select("product").union(sales.select("product")).distinct()
)
inventory = inventory.join(F.broadcast(valid_products), "product", "inner")
# Derived: defensive recompute of the restock flag from on_hand vs reorder point
inventory = inventory.withColumn(
    "restock_alert",
    F.col("on_hand_units") <= F.col("reorder_point_units")
)
# Surrogate key: deterministic SHA-256 over the grain (snapshot_date|warehouse|product)
# Uniquely identifies one stock position; reused by inventory_alerts as a foreign key.
inventory = inventory.withColumn(
    "inventory_key",
    F.sha2(F.concat_ws("||",
        F.date_format(F.col("snapshot_date"), "yyyy-MM-dd"), "warehouse", "product"), 256)
)
# Dedupe composite key (one position per product per warehouse per day)
inventory = dedupe_keep_latest(
    inventory, ["snapshot_date", "product", "warehouse"], "snapshot_date"
)
write_silver(inventory, "inventory_snapshots")

# -------------------------------------------------------------------
# 8. inventory_alerts (manager restock alert view)
# -------------------------------------------------------------------
print("Cleansing inventory_alerts...")
alerts = (
    bronze("inventory_alerts")
    .withColumn("snapshot_date", F.to_date(F.col("snapshot_date"), "yyyy-MM-dd"))
    .withColumn("warehouse", F.trim(F.col("warehouse")))
    .withColumn("product", F.trim(F.col("product")))
    .withColumn("product_family", F.trim(F.col("product_family")))
    .withColumn("on_hand_units", F.col("on_hand_units").cast(IntegerType()))
    .withColumn("reorder_point_units", F.col("reorder_point_units").cast(IntegerType()))
    .withColumn("days_of_supply", F.col("days_of_supply").cast(DoubleType()))
    .withColumn("stock_status", F.upper(F.trim(F.col("stock_status"))))
    .withColumn("recommended_order_units", F.col("recommended_order_units").cast(IntegerType()))
    .filter(F.col("snapshot_date").isNotNull())
    .filter((F.col("product") != "") & (F.col("warehouse") != ""))
    # An alert is only meaningful if it recommends a restock
    .filter(F.col("recommended_order_units") > 0)
    .filter(F.col("stock_status").isin("LOW", "CRITICAL", "OUT"))
)
# Referential integrity: alert must match a cleansed snapshot row
valid_positions = inventory.select(
    "snapshot_date", "warehouse", "product"
).distinct()
alerts = alerts.join(
    F.broadcast(valid_positions), ["snapshot_date", "warehouse", "product"], "inner"
)
# Surrogate key: same SHA-256 over (snapshot_date|warehouse|product) as the snapshot.
# This is BOTH the alert's identifier and its foreign key to inventory_snapshots.
alerts = alerts.withColumn(
    "inventory_key",
    F.sha2(F.concat_ws("||",
        F.date_format(F.col("snapshot_date"), "yyyy-MM-dd"), "warehouse", "product"), 256)
)
# Dedupe composite key
alerts = alerts.dropDuplicates(["snapshot_date", "warehouse", "product"])
write_silver(alerts, "inventory_alerts")

print("\n=== SILVER COMPLETE ===")